# MS-Dialog — Phase 1 Uncertainty Analysis
## Model: `gemini-2.5-flash` | Dataset: MS-Dialog (100 tech-support cases)

**Research question**: When a tech-support model is uncertain, does it ask an *EPISTEMIC* CQ
(reducible knowledge gap) or an *ALEATORIC* CQ (irreducible input ambiguity)?
Does the CQ type predict how much confidence is gained — and does it yield a measurably better solution?

Sections:
1. Setup & Load Data
2. **ROUGE-L Initialisation** — solution quality vs forum accepted answer
3. **Single-Turn Analysis** — confidence, CQ type, Δ confidence × CQ type, ROUGE-L quality, entropy correlations, overconfidence gap
4. **Multi-Turn Analysis** — 4-checkpoint confidence arc, per-turn CQ type, ROUGE-L arc, answer variation
5. **Cross-Experiment Comparison** — confidence + ROUGE-L side by side
6. Export Summary CSV

## 1. Setup

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path("../../").resolve()))

DATASET  = "ms-dialog"
MODEL_ID = "gemini-2.5-flash"
ROOT     = Path("../../").resolve()
OUTPUTS  = ROOT / "outputs" / DATASET / MODEL_ID

import warnings, logging
import numpy as np
import pandas as pd
from IPython.display import display
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.WARNING)
matplotlib.use("Agg")
sns.set_theme(style="whitegrid", palette="muted")
FIGSIZE = (9, 5)

print(f"Model:   {MODEL_ID}")
print(f"Outputs: {OUTPUTS}")
print(f"Exists:  {OUTPUTS.exists()}")

Model:   gemini-2.5-flash
Outputs: D:\final_project\pilot_study\outputs\ms-dialog\gemini-2.5-flash
Exists:  True


## 2. Load Data

In [2]:
# ── Generation results ────────────────────────────────────────────────────────
st_results = pd.read_csv(OUTPUTS / "phase1_singleturn_results.csv")
mt_results = pd.read_csv(OUTPUTS / "phase1_multiturn_results.csv")

# ── Judge labels ──────────────────────────────────────────────────────────────
st_labels  = pd.read_csv(OUTPUTS / "phase1_singleturn_classified.csv")
mt_labels  = pd.read_csv(OUTPUTS / "phase1_multiturn_classified.csv")

# ── Drop blocked rows ─────────────────────────────────────────────────────────
st = st_results[~st_results["was_blocked"]].copy()
mt = mt_results[~mt_results["was_blocked"]].copy()

print(f"Single-turn : {len(st_results)} rows, {int(st_results['was_blocked'].sum())} blocked → {len(st)} usable")
print(f"Multi-turn  : {len(mt_results)} rows, {int(mt_results['was_blocked'].sum())} blocked → {len(mt)} usable")

# ── Resolve id field (dataset uses case_id) ───────────────────────────────────
for df in [st, mt, st_results, mt_results]:
    if "id" not in df.columns and "case_id" in df.columns:
        df["id"] = df["case_id"]

# ── Helpers ───────────────────────────────────────────────────────────────────
def pct(s): return s.mean() * 100

def jaccard(a, b):
    ta = set(str(a).lower().split())
    tb = set(str(b).lower().split())
    if not ta and not tb: return 1.0
    return len(ta & tb) / len(ta | tb)

print(f"\nSingle-turn CQs classified : {len(st_labels)}")
print(f"Multi-turn  CQs classified : {len(mt_labels)}")

Single-turn : 100 rows, 0 blocked → 100 usable
Multi-turn  : 100 rows, 0 blocked → 100 usable

Single-turn CQs classified : 100
Multi-turn  CQs classified : 300


## 2. ROUGE-L Initialisation
*Load accepted answers and compute ROUGE-L F1 (solution quality proxy) at every checkpoint.*

In [3]:
import json
from rouge_score import rouge_scorer as rs

CASES_PATH = ROOT / "datasets" / "ms-dialog" / "msdialog_100.jsonl"
with open(CASES_PATH, encoding="utf-8") as f:
    records_raw = [json.loads(line) for line in f if line.strip()]
accepted = {r.get("case_id") or r.get("id"): r["accepted_answer"] for r in records_raw}
print(f"Loaded {len(accepted)} accepted answers from JSONL")

# ── ROUGE-L scorer ─────────────────────────────────────────────────────────────
_rouge = rs.RougeScorer(["rougeL"], use_stemmer=True)

def rouge_l(hyp, ref):
    """ROUGE-L F1 between hypothesis and reference. Returns NaN if either is empty."""
    if pd.isna(hyp) or pd.isna(ref) or not str(hyp).strip() or not str(ref).strip():
        return np.nan
    return _rouge.score(str(ref), str(hyp))["rougeL"].fmeasure

# ── Single-turn: ROUGE-L at preliminary and updated checkpoints ───────────────
st["accepted_answer"] = st["id"].map(accepted)
st["rouge_prelim"]    = st.apply(lambda r: rouge_l(r["preliminary_solution"], r["accepted_answer"]), axis=1)
st["rouge_updated"]   = st.apply(lambda r: rouge_l(r["updated_solution"],     r["accepted_answer"]), axis=1)
st["rouge_delta"]     = st["rouge_updated"] - st["rouge_prelim"]

print(f"\nSingle-turn ROUGE-L (vs accepted answer):")
print(f"  Preliminary  : {st['rouge_prelim'].mean():.4f}")
print(f"  Updated      : {st['rouge_updated'].mean():.4f}")
print(f"  Δ (mean)     : {st['rouge_delta'].mean():+.4f}")
print(f"  Improved     : {(st['rouge_delta'] > 0).sum()} / {len(st)}")

# ── Multi-turn: ROUGE-L at all 4 checkpoints ─────────────────────────────────
mt["accepted_answer"] = mt["id"].map(accepted)
for new_col, src_col in [
    ("rouge_prelim", "preliminary_solution"),
    ("rouge_s1",     "solution_1"),
    ("rouge_s2",     "solution_2"),
    ("rouge_final",  "final_solution"),
]:
    mt[new_col] = mt.apply(lambda r, c=src_col: rouge_l(r[c], r["accepted_answer"]), axis=1)
mt["rouge_delta"] = mt["rouge_final"] - mt["rouge_prelim"]

rouge_ckpts = [
    ("Preliminary", "rouge_prelim"),
    ("After CQ1",   "rouge_s1"),
    ("After CQ2",   "rouge_s2"),
    ("Final",       "rouge_final"),
]

print(f"\nMulti-turn ROUGE-L (vs accepted answer):")
for lbl, col in rouge_ckpts:
    print(f"  {lbl:<15}: {mt[col].mean():.4f}")
print(f"  Δ (mean)     : {mt['rouge_delta'].mean():+.4f}")
print(f"  Improved     : {(mt['rouge_delta'] > 0).sum()} / {len(mt)}")

Loaded 100 accepted answers from JSONL



Single-turn ROUGE-L (vs accepted answer):
  Preliminary  : 0.1355
  Updated      : 0.1387
  Δ (mean)     : +0.0032
  Improved     : 52 / 100



Multi-turn ROUGE-L (vs accepted answer):
  Preliminary    : 0.1344
  After CQ1      : 0.1349
  After CQ2      : 0.1301
  Final          : 0.1323
  Δ (mean)     : -0.0021
  Improved     : 54 / 100


---
## 3. Single-Turn Analysis
*One CQ round — model generates preliminary solution + CQ, user responds, model gives updated solution*

### 3.1 Confidence Progression (Overall)

In [4]:
prelim_mean  = st["preliminary_confidence"].mean()
updated_mean = st["updated_confidence"].mean()
delta        = st["updated_confidence"] - st["preliminary_confidence"]

print("=== Single-Turn Confidence Progression ===")
print(f"  Preliminary (before CQ) : {prelim_mean:.1f}")
print(f"  Updated     (after CQ)  : {updated_mean:.1f}")
print(f"  Mean delta              : {delta.mean():+.1f}")
print(f"  Median delta            : {delta.median():+.1f}")
print(f"  Std                     : {delta.std():.1f}")
print(f"  Gained confidence       : {(delta > 0).sum()} cases")
print(f"  No change               : {(delta == 0).sum()} cases")
print(f"  Lost confidence         : {(delta < 0).sum()} cases")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Bar: prelim vs updated
axes[0].bar(["Preliminary", "Updated (post-CQ)"], [prelim_mean, updated_mean],
            color=["steelblue", "seagreen"], width=0.45)
axes[0].set_ylabel("Mean Confidence")
axes[0].set_ylim(0, 100)
axes[0].set_title(f"Confidence: Before vs After CQ")
for i, v in enumerate([prelim_mean, updated_mean]):
    axes[0].text(i, v + 1, f"{v:.1f}", ha="center", fontweight="bold")

# Histogram of delta
axes[1].hist(delta, bins=20, color="steelblue", edgecolor="white")
axes[1].axvline(0, color="red", lw=1.5, linestyle="--", label="No change")
axes[1].axvline(delta.mean(), color="orange", lw=1.5, label=f"Mean={delta.mean():+.1f}")
axes[1].set_xlabel("Confidence delta (pp)")
axes[1].set_ylabel("Cases")
axes[1].set_title("Δ Confidence Distribution")
axes[1].legend(fontsize=9)

fig.suptitle(f"Single-Turn Confidence — {MODEL_ID} (n={len(st)})", fontsize=13)
plt.tight_layout()
plt.savefig(OUTPUTS / "st_fig_confidence.png", dpi=150)
plt.show()
print("Saved: st_fig_confidence.png")

=== Single-Turn Confidence Progression ===
  Preliminary (before CQ) : 79.7
  Updated     (after CQ)  : 90.5
  Mean delta              : +10.9
  Median delta            : +10.0
  Std                     : 8.5
  Gained confidence       : 86 cases
  No change               : 11 cases
  Lost confidence         : 3 cases


Saved: st_fig_confidence.png


### 3.2 CQ Type Distribution

In [5]:
q_col_st = "clarifying_question" if "clarifying_question" in st_labels.columns else "question"
st_valid = st_labels[st_labels["label"].isin({"EPISTEMIC", "ALEATORIC"})].copy()

print(f"CQs classified: {len(st_valid)} / {len(st)}")
vc = st_valid["label"].value_counts()
print()
for lbl in ["EPISTEMIC", "ALEATORIC"]:
    n = vc.get(lbl, 0)
    print(f"  {lbl}: {n} ({100*n/len(st_valid):.1f}%)")

fig, ax = plt.subplots(figsize=(5, 4))
colors = {"EPISTEMIC": "steelblue", "ALEATORIC": "salmon"}
ax.bar(vc.index, vc.values, color=[colors.get(l, "gray") for l in vc.index], width=0.45)
ax.set_ylabel("Count")
ax.set_title(f"CQ Type Distribution — {MODEL_ID} (single-turn)")
for i, (lbl, v) in enumerate(vc.items()):
    ax.text(i, v + 0.5, f"{v}\n({100*v/len(st_valid):.0f}%)", ha="center", fontsize=10)
plt.tight_layout()
plt.savefig(OUTPUTS / "st_fig_cq_type.png", dpi=150)
plt.show()
print("Saved: st_fig_cq_type.png")

CQs classified: 100 / 100

  EPISTEMIC: 82 (82.0%)
  ALEATORIC: 18 (18.0%)
Saved: st_fig_cq_type.png


### 3.3 Confidence Delta × CQ Type  ← *Core Research Result*
Does the type of CQ predict how much confidence is gained after the user responds?

In [6]:
# Merge CQ type into single-turn results
# Drop the empty pipeline-written cq_type column first to avoid merge conflicts
st_merge = st.drop(columns=["cq_type"], errors="ignore")
st_with_type = st_merge.merge(
    st_valid[[q_col_st, "label"]].rename(columns={q_col_st: "clarifying_question", "label": "cq_type"}),
    on="clarifying_question", how="left"
)
st_with_type["conf_delta"] = st_with_type["updated_confidence"] - st_with_type["preliminary_confidence"]

typed = st_with_type[st_with_type["cq_type"].isin({"EPISTEMIC", "ALEATORIC"})]

print("=== Confidence Delta by CQ Type ===")
print()
grp = typed.groupby("cq_type")["conf_delta"].agg(["mean", "median", "std", "count"]).round(2)
display(grp.rename(columns={"mean": "Mean Δ", "median": "Median Δ", "std": "Std", "count": "n"}))

# Statistical test: Mann-Whitney U (non-parametric, no normality assumption)
from scipy.stats import mannwhitneyu
ep_vals = typed[typed["cq_type"] == "EPISTEMIC"]["conf_delta"]
al_vals = typed[typed["cq_type"] == "ALEATORIC"]["conf_delta"]
if len(al_vals) >= 2:
    stat, p = mannwhitneyu(ep_vals, al_vals, alternative="two-sided")
    sig = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "n.s."
    print(f"\nMann-Whitney U (EPISTEMIC vs ALEATORIC Δ): U={stat:.0f}, p={p:.4f} {sig}")
else:
    print(f"\nNote: Only {len(al_vals)} ALEATORIC CQ(s) — insufficient for significance test.")

# Visualise
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Box plot
order = ["EPISTEMIC", "ALEATORIC"]
data_plot = [typed[typed["cq_type"] == lbl]["conf_delta"].values for lbl in order]
bp = axes[0].boxplot(data_plot, labels=order, patch_artist=True,
                     medianprops=dict(color="black", lw=2))
for patch, color in zip(bp["boxes"], ["steelblue", "salmon"]):
    patch.set_facecolor(color)
axes[0].axhline(0, color="red", lw=1, linestyle="--")
axes[0].set_ylabel("Confidence delta (pp)")
axes[0].set_title("Δ Confidence by CQ Type")

# Scatter: preliminary_confidence vs conf_delta, coloured by cq_type
for lbl, color in [("EPISTEMIC", "steelblue"), ("ALEATORIC", "salmon")]:
    sub = typed[typed["cq_type"] == lbl]
    axes[1].scatter(sub["preliminary_confidence"], sub["conf_delta"],
                    c=color, label=lbl, alpha=0.7, s=50, edgecolors="none")
axes[1].axhline(0, color="gray", lw=1, linestyle="--")
axes[1].set_xlabel("Preliminary confidence")
axes[1].set_ylabel("Confidence delta (pp)")
axes[1].set_title("Δ Confidence vs Initial Confidence")
axes[1].legend()

fig.suptitle(f"CQ Type × Confidence Delta — {MODEL_ID}", fontsize=13)
plt.tight_layout()
plt.savefig(OUTPUTS / "st_fig_cqtype_delta.png", dpi=150)
plt.show()
print("Saved: st_fig_cqtype_delta.png")

=== Confidence Delta by CQ Type ===



,Mean Δ,Median Δ,Std,n
cq_type,,,,
ALEATORIC,7.50,5.0,5.75,18
EPISTEMIC,11.62,10.0,8.91,82



Mann-Whitney U (EPISTEMIC vs ALEATORIC Δ): U=946, p=0.0575 n.s.


Saved: st_fig_cqtype_delta.png


### 3.4 Confidence Delta by Category
Do some Microsoft product categories show larger confidence gains?

In [7]:
cat_stats = (
    st_with_type
    .groupby("category")["conf_delta"]
    .agg(["mean", "median", "count"])
    .rename(columns={"mean": "Mean Δ", "median": "Median Δ", "count": "n"})
    .sort_values("Mean Δ", ascending=False)
)

print("Confidence delta by product category:")
display(cat_stats.round(1))

fig, ax = plt.subplots(figsize=(12, 5))
cats = cat_stats.index.tolist()
vals = cat_stats["Mean Δ"].values
colors = ["seagreen" if v >= 0 else "salmon" for v in vals]
ax.barh(cats, vals, color=colors)
ax.axvline(0, color="black", lw=0.8)
ax.set_xlabel("Mean Δ Confidence (pp)")
ax.set_title(f"Mean Confidence Delta by Category — {MODEL_ID}")
for i, (v, n) in enumerate(zip(vals, cat_stats["n"].values)):
    ax.text(v + (1 if v >= 0 else -1), i, f"n={n}", va="center", fontsize=8)
plt.tight_layout()
plt.savefig(OUTPUTS / "st_fig_category_delta.png", dpi=150)
plt.show()
print("Saved: st_fig_category_delta.png")

Confidence delta by product category:


,Mean Δ,Median Δ,n
category,,,
Skype_Mac,20.0,20.0,1
Outlook_Email,17.5,17.5,2
Skype_Android,16.7,15.0,3
Windows_10,15.0,15.0,4
Skype_Windows_Desktop,15.0,15.0,2
Games_Windows_10,15.0,15.0,1
Windows_RT_8.1,13.3,20.0,3
Bing_Search,13.0,15.0,5
Skype_iOS,12.6,15.0,5


Saved: st_fig_category_delta.png


### 3.5 CQ Type by Category

In [8]:
typed_cat = st_with_type[st_with_type["cq_type"].isin({"EPISTEMIC", "ALEATORIC"})]
cat_type = typed_cat.groupby(["category", "cq_type"]).size().unstack(fill_value=0)
cat_type["Total"] = cat_type.sum(axis=1)
for lbl in ["EPISTEMIC", "ALEATORIC"]:
    if lbl in cat_type.columns:
        cat_type[f"{lbl} %"] = (cat_type[lbl] / cat_type["Total"] * 100).round(1)

print("CQ type breakdown by category:")
display(cat_type)

CQ type breakdown by category:


cq_type,ALEATORIC,EPISTEMIC,Total,EPISTEMIC %,ALEATORIC %
category,,,,,
Apps_Windows_10,1,7,8,87.5,12.5
Bing,1,5,6,83.3,16.7
Bing_Apps,1,5,6,83.3,16.7
Bing_Maps,1,4,5,80.0,20.0
Bing_Search,0,5,5,100.0,0.0
Excel,5,9,14,64.3,35.7
Games_Windows_10,0,1,1,100.0,0.0
Outlook_Email,0,2,2,100.0,0.0
PowerPoint,5,12,17,70.6,29.4


### 3.6 Solution Quality (ROUGE-L vs Accepted Answer)
ROUGE-L F1 measures lexical overlap between generated solutions and the forum accepted answer — accuracy proxy. Does a CQ improve quality? Does CQ type (EPISTEMIC vs ALEATORIC) predict quality gain? Does confidence gain correlate with quality gain?

In [9]:
# st_with_type already carries rouge cols (inherited from st in cell a0000012)
st_rt    = st_with_type.copy()
typed_rt = st_rt[st_rt["cq_type"].isin({"EPISTEMIC", "ALEATORIC"})]

print("=== Single-Turn Solution Quality (ROUGE-L vs Accepted Answer) ===")
print(f"  Preliminary mean  : {st['rouge_prelim'].mean():.4f}")
print(f"  Updated mean      : {st['rouge_updated'].mean():.4f}")
print(f"  Mean delta        : {st['rouge_delta'].mean():+.4f}")
print(f"  Improved          : {(st['rouge_delta'] > 0).sum()} / {len(st)} cases")

print()
print("ROUGE-L delta by CQ type:")
grp_r = typed_rt.groupby("cq_type")["rouge_delta"].agg(["mean", "median", "count"]).round(4)
display(grp_r.rename(columns={"mean": "Mean Δ ROUGE-L", "median": "Median Δ ROUGE-L", "count": "n"}))

corr_all = st_rt["conf_delta"].corr(st_rt["rouge_delta"])
print(f"\nPearson r(Δ confidence, Δ ROUGE-L) = {corr_all:.3f}")

from scipy.stats import mannwhitneyu as mwu
ep_r = typed_rt[typed_rt["cq_type"] == "EPISTEMIC"]["rouge_delta"]
al_r = typed_rt[typed_rt["cq_type"] == "ALEATORIC"]["rouge_delta"]
if len(al_r) >= 2:
    stat_r, p_r = mwu(ep_r, al_r, alternative="two-sided")
    sig_r = "***" if p_r < 0.001 else "**" if p_r < 0.01 else "*" if p_r < 0.05 else "n.s."
    print(f"Mann-Whitney U (ROUGE-L Δ, EPISTEMIC vs ALEATORIC): p={p_r:.4f} {sig_r}")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 1. Bar: ROUGE-L before vs after CQ
r_bar = [st["rouge_prelim"].mean(), st["rouge_updated"].mean()]
axes[0].bar(["Preliminary", "Updated"], r_bar, color=["steelblue", "seagreen"], width=0.45)
axes[0].set_ylabel("Mean ROUGE-L F1")
axes[0].set_ylim(0, max(r_bar) * 1.35)
axes[0].set_title("ROUGE-L: Before vs After CQ")
for i, v in enumerate(r_bar):
    axes[0].text(i, v + 0.003, f"{v:.3f}", ha="center", fontweight="bold")

# 2. Boxplot: ROUGE-L delta by CQ type
data_r = [typed_rt[typed_rt["cq_type"] == lbl]["rouge_delta"].values
          for lbl in ["EPISTEMIC", "ALEATORIC"]]
bp = axes[1].boxplot(data_r, labels=["EPISTEMIC", "ALEATORIC"], patch_artist=True,
                     medianprops=dict(color="black", lw=2))
for patch, color in zip(bp["boxes"], ["steelblue", "salmon"]):
    patch.set_facecolor(color)
axes[1].axhline(0, color="red", lw=1, linestyle="--")
axes[1].set_ylabel("ROUGE-L delta")
axes[1].set_title("Δ ROUGE-L by CQ Type")

# 3. Scatter: confidence Δ vs ROUGE-L Δ, coloured by CQ type
for lbl, color in [("EPISTEMIC", "steelblue"), ("ALEATORIC", "salmon")]:
    sub = typed_rt[typed_rt["cq_type"] == lbl]
    axes[2].scatter(sub["conf_delta"], sub["rouge_delta"],
                    c=color, label=lbl, alpha=0.75, s=50, edgecolors="none")
axes[2].axhline(0, color="gray", lw=0.8, linestyle="--")
axes[2].axvline(0, color="gray", lw=0.8, linestyle="--")
axes[2].set_xlabel("Confidence delta (pp)")
axes[2].set_ylabel("ROUGE-L delta")
axes[2].set_title(f"Conf Δ vs ROUGE-L Δ  (r={corr_all:.2f})")
axes[2].legend()

fig.suptitle(f"Single-Turn Solution Quality — {MODEL_ID}", fontsize=13)
plt.tight_layout()
plt.savefig(OUTPUTS / "st_fig_rouge.png", dpi=150)
plt.show()
print("Saved: st_fig_rouge.png")

=== Single-Turn Solution Quality (ROUGE-L vs Accepted Answer) ===
  Preliminary mean  : 0.1355
  Updated mean      : 0.1387
  Mean delta        : +0.0032
  Improved          : 52 / 100 cases

ROUGE-L delta by CQ type:


,Mean Δ ROUGE-L,Median Δ ROUGE-L,n
cq_type,,,
ALEATORIC,-0.0026,0.0013,18
EPISTEMIC,0.0045,0.0024,82



Pearson r(Δ confidence, Δ ROUGE-L) = 0.149
Mann-Whitney U (ROUGE-L Δ, EPISTEMIC vs ALEATORIC): p=0.4430 n.s.


Saved: st_fig_rouge.png


### 3.7 Simulator Answer Analysis (Single-Turn)
How often does the user simulator give an informative answer vs a hedge/non-answer?
Responses are classified as **PURE_HEDGE** (≤20 words + hedge phrase), **HEDGED_WITH_INFO** (>20 words + hedge phrase), or **INFORMATIVE**.
We also check whether hedge responses are more likely after ALEATORIC CQs (ambiguity the simulator genuinely cannot resolve).

In [10]:
HEDGE_PHRASES = [
    "not sure", "don't know", "haven't checked", "cannot determine",
    "not provided", "not available", "not mentioned", "not specified",
    "not stated", "not given", "no information", "cannot be determined",
    "i don't", "not enough information", "insufficient information",
    "not directly", "not explicitly", "is not provided", "is not mentioned",
    "cannot tell", "unable to determine", "no details", "i haven't",
    "unsure", "not recorded", "not documented", "not included",
]

def classify_sim_response(text):
    t = str(text).lower()
    hedged = any(h in t for h in HEDGE_PHRASES)
    if hedged:
        return "HEDGED_WITH_INFO" if len(str(text).split()) > 20 else "PURE_HEDGE"
    return "INFORMATIVE"

def is_hedge(text):
    t = str(text).lower()
    return any(h in t for h in HEDGE_PHRASES)

# ── Single-turn simulator analysis ────────────────────────────────────────────
st_sim = st_with_type.copy()
st_sim["sim_class"]      = st_sim["user_response"].apply(classify_sim_response)
st_sim["sim_is_hedge"]   = st_sim["user_response"].apply(is_hedge)
st_sim["conf_delta"]     = st_sim["updated_confidence"] - st_sim["preliminary_confidence"]

n_total = len(st_sim)
vc_sim = st_sim["sim_class"].value_counts()

print("=== Single-Turn Simulator Response Quality ===")
print(f"\nResponse classification (n={n_total}):")
for cls in ["INFORMATIVE", "HEDGED_WITH_INFO", "PURE_HEDGE"]:
    n = vc_sim.get(cls, 0)
    print(f"  {cls:<20}: {n:>3} ({100*n/n_total:.1f}%)")
print(f"\nOverall hedge rate: {st_sim['sim_is_hedge'].mean()*100:.1f}%")

# Confidence delta by response class
print("\nConfidence delta by response class:")
sim_grp = st_sim.groupby("sim_class")["conf_delta"].agg(["mean","median","count"]).round(2)
display(sim_grp.rename(columns={"mean":"Mean Δ conf","median":"Median Δ conf","count":"n"}))

# Hedge rate by CQ type
print("\nHedge rate by CQ type:")
hr_by_type = st_sim[st_sim["cq_type"].isin({"EPISTEMIC","ALEATORIC"})].groupby("cq_type")["sim_is_hedge"].mean()
for lbl, rate in hr_by_type.items():
    print(f"  {lbl}: {rate*100:.1f}%")

# Visualise
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Bar: class distribution
classes = ["INFORMATIVE","HEDGED_WITH_INFO","PURE_HEDGE"]
counts  = [vc_sim.get(c, 0) for c in classes]
colors  = ["seagreen","steelblue","salmon"]
axes[0].bar(classes, counts, color=colors, edgecolor="white")
axes[0].set_ylabel("Count")
axes[0].set_title("Simulator Response Classes (ST)")
axes[0].tick_params(axis="x", rotation=15)
for i, (c, n) in enumerate(zip(classes, counts)):
    axes[0].text(i, n+0.5, f"{n}\n({100*n/n_total:.0f}%)", ha="center", fontsize=9)

# Box: conf delta by class
data_cls = [st_sim[st_sim["sim_class"]==c]["conf_delta"].values for c in classes]
bp = axes[1].boxplot(data_cls, labels=classes, patch_artist=True,
                     medianprops=dict(color="black", lw=2))
for patch, color in zip(bp["boxes"], colors):
    patch.set_facecolor(color)
axes[1].axhline(0, color="red", lw=1, linestyle="--")
axes[1].set_ylabel("Confidence delta (pp)")
axes[1].set_title("Conf Delta by Response Class")
axes[1].tick_params(axis="x", rotation=15)

# Bar: hedge rate by CQ type
typed_sim = st_sim[st_sim["cq_type"].isin({"EPISTEMIC","ALEATORIC"})]
hr_vals = [typed_sim[typed_sim["cq_type"]==lbl]["sim_is_hedge"].mean()*100
           for lbl in ["EPISTEMIC","ALEATORIC"]]
bars = axes[2].bar(["EPISTEMIC","ALEATORIC"], hr_vals,
                   color=["steelblue","salmon"], width=0.4)
axes[2].set_ylabel("Hedge rate (%)")
axes[2].set_ylim(0, 110)
axes[2].set_title("Simulator Hedge Rate by CQ Type")
for bar, v in zip(bars, hr_vals):
    axes[2].text(bar.get_x()+bar.get_width()/2, v+2, f"{v:.1f}%",
                 ha="center", fontweight="bold")

fig.suptitle(f"Simulator Answer Analysis — Single-Turn ({MODEL_ID})", fontsize=13)
plt.tight_layout()
plt.savefig(OUTPUTS / "st_fig_simulator.png", dpi=150)
plt.show()
print("Saved: st_fig_simulator.png")


=== Single-Turn Simulator Response Quality ===

Response classification (n=100):
  INFORMATIVE         :  30 (30.0%)
  HEDGED_WITH_INFO    :  26 (26.0%)
  PURE_HEDGE          :  44 (44.0%)

Overall hedge rate: 70.0%

Confidence delta by response class:


,Mean Δ conf,Median Δ conf,n
sim_class,,,
HEDGED_WITH_INFO,10.88,10.0,26
INFORMATIVE,11.00,10.0,30
PURE_HEDGE,10.80,10.0,44



Hedge rate by CQ type:
  ALEATORIC: 50.0%
  EPISTEMIC: 74.4%


Saved: st_fig_simulator.png


### 3.8 Entropy Correlation (Spearman / Pearson)
**Token entropy** (word-level Shannon entropy of the preliminary solution) is a proxy for distributional uncertainty — when the model cannot commit to one interpretation it tends to produce more varied vocabulary.

**Semantic entropy** (1 − ROUGE-L vs accepted answer) is a proxy for meaning-level uncertainty — how far the model's answer is from the ground truth.

Hypothesis: token entropy correlates with **ALEATORIC** labels (input ambiguity → diverse output); semantic entropy correlates with **EPISTEMIC** labels (knowledge gap → wrong answer).

In [11]:
from scipy.stats import entropy as sci_entropy, spearmanr, pearsonr
from collections import Counter

def token_entropy_norm(text):
    """Normalised word-level Shannon entropy (0=uniform single token, 1=max diversity)."""
    words = str(text).lower().split()
    if len(words) < 2:
        return 0.0
    counts = Counter(words)
    probs   = [c / len(words) for c in counts.values()]
    h       = sci_entropy(probs, base=2)
    max_h   = np.log2(len(counts)) if len(counts) > 1 else 1.0
    return float(h / max_h) if max_h > 0 else 0.0

ent_df = typed_rt.copy()
ent_df["token_entropy"]    = ent_df["preliminary_solution"].apply(token_entropy_norm)
ent_df["semantic_entropy"] = 1.0 - ent_df["rouge_prelim"]   # low ROUGE-L = high semantic uncertainty
ent_df["is_aleatoric"]     = (ent_df["cq_type"] == "ALEATORIC").astype(float)
ent_df["is_epistemic"]     = (ent_df["cq_type"] == "EPISTEMIC").astype(float)

print("=== Entropy × CQ Type Correlations ===\n")
results = {}
for metric, label_col, hyp_label in [
    ("token_entropy",    "is_aleatoric", "ALEATORIC"),
    ("semantic_entropy", "is_epistemic", "EPISTEMIC"),
]:
    clean  = ent_df[[metric, label_col]].dropna()
    sp_r, sp_p = spearmanr(clean[metric], clean[label_col])
    pe_r, pe_p = pearsonr( clean[metric], clean[label_col])
    sp_sig = "***" if sp_p<0.001 else "**" if sp_p<0.01 else "*" if sp_p<0.05 else "n.s."
    pe_sig = "***" if pe_p<0.001 else "**" if pe_p<0.01 else "*" if pe_p<0.05 else "n.s."
    results[metric] = dict(sp_r=sp_r, sp_p=sp_p, sp_sig=sp_sig,
                           pe_r=pe_r, pe_p=pe_p, pe_sig=pe_sig)
    print(f"  {metric}  ↔  {hyp_label} label:")
    print(f"    Spearman ρ = {sp_r:+.3f}   p = {sp_p:.4f}  {sp_sig}")
    print(f"    Pearson  r = {pe_r:+.3f}   p = {pe_p:.4f}  {pe_sig}")
    print()

print("Mean token entropy by CQ type:")
display(ent_df.groupby("cq_type")["token_entropy"].agg(["mean","std"]).round(3))
print("\nMean semantic entropy (1-ROUGE-L) by CQ type:")
display(ent_df.groupby("cq_type")["semantic_entropy"].agg(["mean","std"]).round(3))

# Visualise
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, metric, label_col, lbl_name, color in [
    (axes[0], "token_entropy",    "is_aleatoric", "ALEATORIC", "salmon"),
    (axes[1], "semantic_entropy", "is_epistemic", "EPISTEMIC", "steelblue"),
]:
    for cq_lbl, c in [("EPISTEMIC","steelblue"),("ALEATORIC","salmon")]:
        sub = ent_df[ent_df["cq_type"]==cq_lbl]
        ax.scatter(sub[metric], sub[label_col]+np.random.uniform(-0.05,0.05,len(sub)),
                   c=c, alpha=0.5, s=30, label=cq_lbl)
    r = results[metric]
    ax.set_xlabel(metric.replace("_"," ").title())
    ax.set_ylabel(f"Is {lbl_name} (0/1)")
    ax.set_title(f"{metric.replace('_',' ').title()} ↔ {lbl_name}\n"
                 f"Spearman ρ={r['sp_r']:+.3f} {r['sp_sig']}  "
                 f"Pearson r={r['pe_r']:+.3f} {r['pe_sig']}")
    ax.legend(fontsize=8)

fig.suptitle(f"Entropy Correlations — {MODEL_ID} (n={len(ent_df)})", fontsize=13)
plt.tight_layout()
plt.savefig(OUTPUTS / "st_fig_entropy_corr.png", dpi=150)
plt.show()
print("Saved: st_fig_entropy_corr.png")


=== Entropy × CQ Type Correlations ===

  token_entropy  ↔  ALEATORIC label:
    Spearman ρ = -0.068   p = 0.5009  n.s.
    Pearson  r = -0.072   p = 0.4735  n.s.

  semantic_entropy  ↔  EPISTEMIC label:
    Spearman ρ = +0.271   p = 0.0065  **
    Pearson  r = +0.171   p = 0.0892  n.s.

Mean token entropy by CQ type:


,mean,std
cq_type,,
ALEATORIC,0.972,0.013
EPISTEMIC,0.975,0.012



Mean semantic entropy (1-ROUGE-L) by CQ type:


,mean,std
cq_type,,
ALEATORIC,0.842,0.047
EPISTEMIC,0.869,0.063


Saved: st_fig_entropy_corr.png


### 3.9 Overconfidence Gap Analysis
**Overconfidence gap** = `confidence / 100 − ROUGE-L` — positive values mean the model is more confident than its solution quality warrants.

We analyse this at the preliminary (pre-CQ) and updated (post-CQ) checkpoints, and check whether the CQ reduces the gap (i.e. confidence becomes better calibrated after the user responds).

We also produce an **expected calibration error (ECE)** approximation by binning confidence and computing mean ROUGE-L per bin.

In [12]:
oc_df = st_with_type.copy()
oc_df["gap_prelim"]  = oc_df["preliminary_confidence"] / 100.0 - oc_df["rouge_prelim"]
oc_df["gap_updated"] = oc_df["updated_confidence"]     / 100.0 - oc_df["rouge_updated"]
oc_df["gap_delta"]   = oc_df["gap_updated"] - oc_df["gap_prelim"]   # negative = better calibrated

def cal_class(gap):
    if gap > 0.15:  return "Overconfident"
    if gap < -0.15: return "Underconfident"
    return "Well-calibrated"

oc_df["cal_class_prelim"]  = oc_df["gap_prelim"].apply(cal_class)
oc_df["cal_class_updated"] = oc_df["gap_updated"].apply(cal_class)

print("=== Overconfidence Gap Analysis ===\n")
print(f"Preliminary gap  — mean: {oc_df['gap_prelim'].mean():+.3f}  "
      f"median: {oc_df['gap_prelim'].median():+.3f}")
print(f"Updated gap      — mean: {oc_df['gap_updated'].mean():+.3f}  "
      f"median: {oc_df['gap_updated'].median():+.3f}")
print(f"Gap delta        — mean: {oc_df['gap_delta'].mean():+.3f}  "
      f"(negative = improved calibration)")

print("\nCalibration class — Preliminary:")
display(oc_df["cal_class_prelim"].value_counts())
print("\nCalibration class — Updated:")
display(oc_df["cal_class_updated"].value_counts())

# ECE approximation: bin confidence, compute mean ROUGE-L per bin
bins = [0, 50, 60, 70, 80, 90, 100]
oc_df["conf_bin"]   = pd.cut(oc_df["preliminary_confidence"], bins=bins, right=True)
oc_df["conf_bin_u"] = pd.cut(oc_df["updated_confidence"],     bins=bins, right=True)
ece_prelim  = oc_df.groupby("conf_bin",  observed=True)[["preliminary_confidence","rouge_prelim"]].mean()
ece_updated = oc_df.groupby("conf_bin_u",observed=True)[["updated_confidence","rouge_updated"]].mean()
ece_prelim["calibration_gap"]  = ece_prelim["preliminary_confidence"]/100 - ece_prelim["rouge_prelim"]
ece_updated["calibration_gap"] = ece_updated["updated_confidence"]/100    - ece_updated["rouge_updated"]

print("\nECE bins — Preliminary (conf/100 vs ROUGE-L):")
display(ece_prelim.round(3))

# Overconfidence gap by CQ type
oc_typed = oc_df[oc_df["cq_type"].isin({"EPISTEMIC","ALEATORIC"})]
print("\nOverconfidence gap (preliminary) by CQ type:")
display(oc_typed.groupby("cq_type")["gap_prelim"].agg(["mean","median","std"]).round(3))

# Visualise
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 1. Scatter: confidence vs ROUGE-L (pre-CQ), diagonal = perfect calibration
for lbl, color in [("EPISTEMIC","steelblue"),("ALEATORIC","salmon")]:
    sub = oc_typed[oc_typed["cq_type"]==lbl]
    axes[0].scatter(sub["preliminary_confidence"]/100, sub["rouge_prelim"],
                    c=color, alpha=0.65, s=40, label=lbl)
axes[0].plot([0,1],[0,1], "k--", lw=1, label="Perfect calibration")
axes[0].set_xlabel("Confidence (normalised)")
axes[0].set_ylabel("ROUGE-L (quality proxy)")
axes[0].set_title("Confidence vs Quality (Pre-CQ)")
axes[0].legend(fontsize=8)

# 2. Hist of overconfidence gap before/after CQ
axes[1].hist(oc_df["gap_prelim"],  bins=20, alpha=0.6, color="steelblue",
             label=f"Preliminary  (μ={oc_df['gap_prelim'].mean():+.3f})", edgecolor="white")
axes[1].hist(oc_df["gap_updated"], bins=20, alpha=0.6, color="seagreen",
             label=f"Updated      (μ={oc_df['gap_updated'].mean():+.3f})", edgecolor="white")
axes[1].axvline(0, color="red", lw=1.5, linestyle="--", label="Perfect calibration")
axes[1].set_xlabel("Overconfidence gap (conf/100 − ROUGE-L)")
axes[1].set_ylabel("Count")
axes[1].set_title("Overconfidence Gap Distribution")
axes[1].legend(fontsize=8)

# 3. ECE plot: conf bin midpoint vs mean ROUGE-L
prelim_conf_mid = ece_prelim["preliminary_confidence"].values / 100
prelim_rouge    = ece_prelim["rouge_prelim"].values
upd_conf_mid    = ece_updated["updated_confidence"].values / 100
upd_rouge       = ece_updated["rouge_updated"].values
axes[2].plot([0,1],[0,1],"k--", lw=1, label="Perfect calibration")
axes[2].plot(prelim_conf_mid, prelim_rouge, "o-", color="steelblue",
             lw=2, ms=7, label="Pre-CQ")
axes[2].plot(upd_conf_mid,   upd_rouge,    "s-", color="seagreen",
             lw=2, ms=7, label="Post-CQ")
axes[2].set_xlabel("Mean Confidence (bin midpoint, normalised)")
axes[2].set_ylabel("Mean ROUGE-L")
axes[2].set_title("ECE Calibration Plot")
axes[2].legend(fontsize=8)

fig.suptitle(f"Overconfidence Gap Analysis — {MODEL_ID}", fontsize=13)
plt.tight_layout()
plt.savefig(OUTPUTS / "st_fig_overconfidence.png", dpi=150)
plt.show()
print("Saved: st_fig_overconfidence.png")


=== Overconfidence Gap Analysis ===

Preliminary gap  — mean: +0.661  median: +0.676
Updated gap      — mean: +0.767  median: +0.769
Gap delta        — mean: +0.106  (negative = improved calibration)

Calibration class — Preliminary:


cal_class_prelim
Overconfident    100
Name: count, dtype: int64


Calibration class — Updated:


cal_class_updated
Overconfident    100
Name: count, dtype: int64


ECE bins — Preliminary (conf/100 vs ROUGE-L):


,preliminary_confidence,rouge_prelim,calibration_gap
conf_bin,,,
"(50, 60]",60.000,0.130,0.470
"(60, 70]",66.944,0.106,0.563
"(70, 80]",75.972,0.114,0.646
"(80, 90]",87.568,0.165,0.711
"(90, 100]",95.000,0.168,0.782



Overconfidence gap (preliminary) by CQ type:


,mean,median,std
cq_type,,,
ALEATORIC,0.701,0.715,0.063
EPISTEMIC,0.652,0.665,0.089


Saved: st_fig_overconfidence.png


---
## 4. Multi-Turn Analysis
*Three CQ rounds — 4 confidence checkpoints: Preliminary → After CQ1 → After CQ2 → Final*

### 4.1 Confidence Progression Across Checkpoints

In [13]:
mt_ckpts = [
    ("Preliminary",  "preliminary_confidence"),
    ("After CQ1",    "confidence_1"),
    ("After CQ2",    "confidence_2"),
    ("Final",        "final_confidence"),
]

print("=== Multi-Turn Confidence Progression ===")
print(f"{'Checkpoint':<15} {'Mean':>7} {'Median':>8} {'Std':>7}")
print("-" * 40)
means = []
for label, col in mt_ckpts:
    m = mt[col].mean()
    med = mt[col].median()
    s = mt[col].std()
    means.append(m)
    print(f"  {label:<13} {m:>7.1f} {med:>8.1f} {s:>7.1f}")
print(f"\n  Total gain (Preliminary → Final): {means[-1] - means[0]:+.1f} pp")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Line: mean confidence at each checkpoint
labels = [l for l, _ in mt_ckpts]
axes[0].plot(labels, means, "o-", lw=2, ms=8, color="steelblue")
for i, (lbl, m) in enumerate(zip(labels, means)):
    axes[0].text(i, m + 0.5, f"{m:.1f}", ha="center", fontsize=9)
axes[0].set_ylabel("Mean Confidence")
axes[0].set_ylim(0, 100)
axes[0].set_title("Mean Confidence at Each Checkpoint")
axes[0].tick_params(axis="x", rotation=15)

# Individual case trajectories (thin grey) + mean (thick blue)
cols = [c for _, c in mt_ckpts]
for _, row in mt.iterrows():
    axes[1].plot(range(4), [row[c] for c in cols], color="gray", alpha=0.15, lw=0.8)
axes[1].plot(range(4), means, "o-", lw=2.5, ms=8, color="steelblue", label="Mean", zorder=5)
axes[1].set_xticks(range(4))
axes[1].set_xticklabels(labels, rotation=15)
axes[1].set_ylabel("Confidence")
axes[1].set_ylim(0, 100)
axes[1].set_title("Individual Case Trajectories")
axes[1].legend()

fig.suptitle(f"Multi-Turn Confidence Progression — {MODEL_ID} (n={len(mt)})", fontsize=13)
plt.tight_layout()
plt.savefig(OUTPUTS / "mt_fig_confidence_arc.png", dpi=150)
plt.show()
print("Saved: mt_fig_confidence_arc.png")

=== Multi-Turn Confidence Progression ===
Checkpoint         Mean   Median     Std
----------------------------------------
  Preliminary      78.3     75.0     9.1
  After CQ1        83.6     85.0     8.4
  After CQ2        85.0     85.0     9.7
  Final            92.1     95.0     7.9

  Total gain (Preliminary → Final): +13.8 pp


Saved: mt_fig_confidence_arc.png


### 4.2 CQ Type Distribution Per Turn

In [14]:
q_col_mt = "clarifying_question" if "clarifying_question" in mt_labels.columns else "question"
mt_valid = mt_labels[mt_labels["label"].isin({"EPISTEMIC", "ALEATORIC"})].copy()

# mt_labels should have 'turn' column (judge writes id, turn, clarifying_question, label)
if "turn" not in mt_valid.columns:
    print("Warning: 'turn' column not found in mt_labels — building from results")
    long_rows = []
    for _, r in mt.iterrows():
        for t in range(1, 4):
            long_rows.append({"id": r["id"], "turn": t, q_col_mt: r[f"cq_{t}"]})
    long_df = pd.DataFrame(long_rows)
    mt_valid = mt_valid.merge(long_df, on=["id", q_col_mt], how="left")

print(f"Multi-turn CQs classified: {len(mt_valid)} (expected {len(mt)*3})")
print()

# Overall distribution
vc_mt = mt_valid["label"].value_counts()
for lbl in ["EPISTEMIC", "ALEATORIC"]:
    n = vc_mt.get(lbl, 0)
    print(f"  Overall {lbl}: {n} ({100*n/len(mt_valid):.1f}%)")

print()
print("By turn:")
turn_dist = mt_valid.groupby(["turn", "label"]).size().unstack(fill_value=0)
display(turn_dist)

# Plot CQ type per turn
turns = [1, 2, 3]
ep_pct = []
al_pct = []
for t in turns:
    sub = mt_valid[mt_valid["turn"] == t]
    total = len(sub)
    ep_pct.append(100 * sub[sub["label"]=="EPISTEMIC"].shape[0] / total if total else 0)
    al_pct.append(100 * sub[sub["label"]=="ALEATORIC"].shape[0] / total if total else 0)

fig, ax = plt.subplots(figsize=(6, 4))
x = np.arange(len(turns))
ax.bar(x - 0.18, ep_pct, 0.35, label="EPISTEMIC", color="steelblue")
ax.bar(x + 0.18, al_pct, 0.35, label="ALEATORIC", color="salmon")
ax.set_xticks(x)
ax.set_xticklabels([f"CQ{t}" for t in turns])
ax.set_ylabel("% of CQs")
ax.set_ylim(0, 115)
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.set_title(f"CQ Type by Turn — {MODEL_ID}")
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUTS / "mt_fig_cq_type_per_turn.png", dpi=150)
plt.show()
print("Saved: mt_fig_cq_type_per_turn.png")

Multi-turn CQs classified: 300 (expected 300)

  Overall EPISTEMIC: 256 (85.3%)
  Overall ALEATORIC: 44 (14.7%)

By turn:


label,ALEATORIC,EPISTEMIC
turn,,
1,16,84
2,15,85
3,13,87


Saved: mt_fig_cq_type_per_turn.png


### 4.3 Confidence Delta × CQ Type Per Turn  ← *Core Research Result*
Does asking an EPISTEMIC CQ at turn T lead to a larger confidence gain at turn T+1?

In [15]:
from scipy.stats import mannwhitneyu

# Build per-turn delta table
conf_cols = [
    (1, "preliminary_confidence", "confidence_1"),
    (2, "confidence_1",           "confidence_2"),
    (3, "confidence_2",           "final_confidence"),
]

rows = []
for _, r in mt.iterrows():
    for turn_idx, c_before, c_after in conf_cols:
        cq_text = r.get(f"cq_{turn_idx}", "")
        label_rows = mt_valid[(mt_valid["id"] == r["id"]) & (mt_valid["turn"] == turn_idx)]
        lbl = label_rows["label"].iloc[0] if len(label_rows) else None
        rows.append({
            "id": r["id"],
            "turn": turn_idx,
            "cq_type": lbl,
            "conf_before": r[c_before],
            "conf_after":  r[c_after],
            "conf_delta":  r[c_after] - r[c_before],
        })

turn_df = pd.DataFrame(rows)
typed_turn = turn_df[turn_df["cq_type"].isin({"EPISTEMIC", "ALEATORIC"})]

print("=== Confidence Delta by Turn × CQ Type ===")
print()
for t in [1, 2, 3]:
    sub = typed_turn[typed_turn["turn"] == t]
    ep = sub[sub["cq_type"]=="EPISTEMIC"]["conf_delta"]
    al = sub[sub["cq_type"]=="ALEATORIC"]["conf_delta"]
    print(f"  Turn {t}:  EPISTEMIC mean={ep.mean():+.1f} (n={len(ep)})  "
          f"| ALEATORIC mean={al.mean():+.1f} (n={len(al)})")
    if len(al) >= 2 and len(ep) >= 2:
        stat, p = mannwhitneyu(ep, al, alternative="two-sided")
        sig = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "n.s."
        print(f"           Mann-Whitney: p={p:.4f} {sig}")
    else:
        print(f"           (insufficient ALEATORIC cases for test)")

print()
print("Pooled across all turns:")
ep_all = typed_turn[typed_turn["cq_type"]=="EPISTEMIC"]["conf_delta"]
al_all = typed_turn[typed_turn["cq_type"]=="ALEATORIC"]["conf_delta"]
print(f"  EPISTEMIC: mean={ep_all.mean():+.1f}, median={ep_all.median():+.1f}, n={len(ep_all)}")
print(f"  ALEATORIC: mean={al_all.mean():+.1f}, median={al_all.median():+.1f}, n={len(al_all)}")
if len(al_all) >= 2:
    stat, p = mannwhitneyu(ep_all, al_all, alternative="two-sided")
    sig = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "n.s."
    print(f"  Mann-Whitney (pooled): p={p:.4f} {sig}")

# Visualise per-turn delta by type
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)
for ax, t in zip(axes, [1, 2, 3]):
    sub = typed_turn[typed_turn["turn"] == t]
    data = [sub[sub["cq_type"]==lbl]["conf_delta"].values for lbl in ["EPISTEMIC", "ALEATORIC"]]
    bp = ax.boxplot(data, labels=["EPISTEMIC", "ALEATORIC"], patch_artist=True,
                    medianprops=dict(color="black", lw=2))
    for patch, color in zip(bp["boxes"], ["steelblue", "salmon"]):
        patch.set_facecolor(color)
    ax.axhline(0, color="red", lw=1, linestyle="--")
    ax.set_title(f"CQ{t}")
    if t == 1:
        ax.set_ylabel("Confidence delta (pp)")

fig.suptitle(f"Conf Delta by CQ Type per Turn — {MODEL_ID}", fontsize=13)
plt.tight_layout()
plt.savefig(OUTPUTS / "mt_fig_cqtype_delta_per_turn.png", dpi=150)
plt.show()
print("Saved: mt_fig_cqtype_delta_per_turn.png")

=== Confidence Delta by Turn × CQ Type ===

  Turn 1:  EPISTEMIC mean=+5.3 (n=84)  | ALEATORIC mean=+5.4 (n=16)
           Mann-Whitney: p=0.8711 n.s.
  Turn 2:  EPISTEMIC mean=+1.2 (n=85)  | ALEATORIC mean=+2.4 (n=15)
           Mann-Whitney: p=0.4337 n.s.
  Turn 3:  EPISTEMIC mean=+7.3 (n=87)  | ALEATORIC mean=+5.7 (n=13)
           Mann-Whitney: p=0.1518 n.s.

Pooled across all turns:
  EPISTEMIC: mean=+4.6, median=+5.0, n=256
  ALEATORIC: mean=+4.5, median=+5.0, n=44
  Mann-Whitney (pooled): p=0.9021 n.s.


Saved: mt_fig_cqtype_delta_per_turn.png


### 4.4 Cross-Turn CQ Type Consistency
Does the model consistently choose EPISTEMIC vs ALEATORIC across turns for the same case?

In [16]:
# Pivot to wide format: one row per case, columns cq1_type, cq2_type, cq3_type
wide = mt_valid.pivot_table(index="id", columns="turn", values="label", aggfunc="first")
wide.columns = [f"cq{t}_type" for t in wide.columns]
wide = wide.reset_index()

# Cases where all 3 CQs are the same type
if "cq1_type" in wide.columns and "cq2_type" in wide.columns and "cq3_type" in wide.columns:
    all_same = wide.apply(lambda r: r["cq1_type"]==r["cq2_type"]==r["cq3_type"], axis=1)
    print(f"Cases with same CQ type all 3 turns: {all_same.sum()}/{len(wide)} ({100*all_same.mean():.1f}%)")
    print()
    # Pattern distribution
    if all(c in wide.columns for c in ["cq1_type", "cq2_type", "cq3_type"]):
        patterns = wide[["cq1_type","cq2_type","cq3_type"]].apply(
            lambda r: "→".join([str(r[c])[0] for c in ["cq1_type","cq2_type","cq3_type"]]), axis=1
        ).value_counts()
        print("CQ type sequence patterns (E=EPISTEMIC, A=ALEATORIC):")
        for pat, n in patterns.items():
            print(f"  {pat.replace('E', 'E').replace('A', 'A')}: {n} cases")
else:
    print("Columns:", wide.columns.tolist())

Cases with same CQ type all 3 turns: 81/100 (81.0%)

CQ type sequence patterns (E=EPISTEMIC, A=ALEATORIC):
  E→E→E: 75 cases
  A→E→E: 6 cases
  A→A→A: 6 cases
  E→A→E: 4 cases
  E→A→A: 3 cases
  A→E→A: 2 cases
  A→A→E: 2 cases
  E→E→A: 2 cases


### 4.5 CQ Lexical Diversity (Jaccard)
Are the CQs across turns asking different things?

In [17]:
print("Mean token Jaccard similarity between CQ pairs (lower = more diverse):")
for ca, cb in [("cq_1","cq_2"), ("cq_2","cq_3"), ("cq_1","cq_3")]:
    sims = mt.apply(lambda r: jaccard(r[ca], r[cb]), axis=1)
    print(f"  {ca} vs {cb}: {sims.mean():.3f}  (median {sims.median():.3f})")

Mean token Jaccard similarity between CQ pairs (lower = more diverse):
  cq_1 vs cq_2: 0.160  (median 0.143)
  cq_2 vs cq_3: 0.166  (median 0.152)
  cq_1 vs cq_3: 0.142  (median 0.129)


### 4.6 Simulator Response Quality
How often does the simulator give an informative answer vs. a non-answer?

In [18]:
print("=== Multi-Turn Simulator Response Quality ===\n")

mt_sim = mt.copy()
for t in [1, 2, 3]:
    col = f"user_response_{t}"
    mt_sim[f"sim_class_{t}"]    = mt_sim[col].fillna("").apply(classify_sim_response)
    mt_sim[f"sim_is_hedge_{t}"] = mt_sim[col].fillna("").apply(is_hedge)

print(f"{'Turn':<6} {'INFORMATIVE':>14} {'HEDGED+INFO':>14} {'PURE_HEDGE':>12} {'Hedge%':>8}")
print("-" * 60)
for t in [1, 2, 3]:
    cls_col = f"sim_class_{t}"
    _cls_vc = mt_sim[cls_col].value_counts()
    n_info  = _cls_vc.get("INFORMATIVE", 0)
    n_hi    = _cls_vc.get("HEDGED_WITH_INFO", 0)
    n_ph    = _cls_vc.get("PURE_HEDGE", 0)
    hedge_r = mt_sim[f"sim_is_hedge_{t}"].mean() * 100
    print(f"  CQ{t}  {n_info:>14} ({100*n_info/len(mt_sim):.0f}%)"
          f"  {n_hi:>9} ({100*n_hi/len(mt_sim):.0f}%)"
          f"  {n_ph:>7} ({100*n_ph/len(mt_sim):.0f}%)"
          f"  {hedge_r:>6.1f}%")

# Hedge rate trend across turns
hedge_rates = [mt_sim[f"sim_is_hedge_{t}"].mean()*100 for t in [1,2,3]]
print(f"\nHedge rate trend: CQ1={hedge_rates[0]:.1f}%  ->  CQ2={hedge_rates[1]:.1f}%  ->  CQ3={hedge_rates[2]:.1f}%")

# Hedge rate by CQ type per turn
print("\nHedge rate by CQ type per turn:")
for t in [1,2,3]:
    turn_types = mt_valid[mt_valid["turn"]==t][["id","label"]].rename(columns={"label":"cq_type_t"})
    sub = mt_sim.merge(turn_types, on="id", how="left")
    hr = sub.groupby("cq_type_t")[f"sim_is_hedge_{t}"].mean().map("{:.1%}".format)
    print(f"  CQ{t}: {hr.to_dict()}")

# Confidence gain by response class per turn
print("\nConf delta (turn) when simulator gives INFORMATIVE vs HEDGE:")
conf_seq = [(1,"preliminary_confidence","confidence_1"),
            (2,"confidence_1","confidence_2"),
            (3,"confidence_2","final_confidence")]
for t, c_before, c_after in conf_seq:
    mt_sim[f"conf_delta_{t}"] = mt_sim[c_after] - mt_sim[c_before]
    grp = mt_sim.groupby(f"sim_class_{t}")[f"conf_delta_{t}"].mean().round(2)
    print(f"  CQ{t}: {grp.to_dict()}")

# Visualise
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Stacked bar: class distribution per turn
turns = [1, 2, 3]
inf_pct = [mt_sim[f"sim_class_{t}"].eq("INFORMATIVE").mean()*100      for t in turns]
hi_pct  = [mt_sim[f"sim_class_{t}"].eq("HEDGED_WITH_INFO").mean()*100 for t in turns]
ph_pct  = [mt_sim[f"sim_class_{t}"].eq("PURE_HEDGE").mean()*100        for t in turns]
x = np.arange(len(turns))
axes[0].bar(x, inf_pct, label="INFORMATIVE",  color="seagreen")
axes[0].bar(x, hi_pct,  bottom=inf_pct,        label="HEDGED+INFO", color="steelblue")
bottom2 = [i+h for i,h in zip(inf_pct, hi_pct)]
axes[0].bar(x, ph_pct,  bottom=bottom2,         label="PURE_HEDGE",  color="salmon")
axes[0].set_xticks(x)
axes[0].set_xticklabels([f"CQ{t}" for t in turns])
axes[0].set_ylabel("% of responses")
axes[0].set_ylim(0, 115)
axes[0].set_title("Simulator Response Class per Turn (MT)")
axes[0].legend(fontsize=9)
for i, ip in enumerate(inf_pct):
    axes[0].text(i, ip/2, f"{ip:.0f}%", ha="center", va="center",
                 fontsize=8, color="white", fontweight="bold")

# Hedge rate line
axes[1].plot(turns, hedge_rates, "o-", lw=2.5, ms=8, color="salmon")
for t, v in zip(turns, hedge_rates):
    axes[1].text(t, v+1.5, f"{v:.1f}%", ha="center", fontsize=9)
axes[1].set_xticks(turns)
axes[1].set_xticklabels([f"CQ{t}" for t in turns])
axes[1].set_ylim(0, 110)
axes[1].set_ylabel("Hedge rate (%)")
axes[1].set_title("Hedge Rate Across Turns")
axes[1].axhline(50, color="gray", lw=0.8, linestyle="--", label="50% reference")

fig.suptitle(f"Simulator Answer Analysis - Multi-Turn ({MODEL_ID})", fontsize=13)
plt.tight_layout()
plt.savefig(OUTPUTS / "mt_fig_simulator.png", dpi=150)
plt.show()
print("Saved: mt_fig_simulator.png")


=== Multi-Turn Simulator Response Quality ===

Turn      INFORMATIVE    HEDGED+INFO   PURE_HEDGE   Hedge%
------------------------------------------------------------
  CQ1              32 (32%)         36 (36%)       32 (32%)    68.0%
  CQ2              13 (13%)         38 (38%)       49 (49%)    87.0%
  CQ3              12 (12%)         34 (34%)       54 (54%)    88.0%

Hedge rate trend: CQ1=68.0%  ->  CQ2=87.0%  ->  CQ3=88.0%

Hedge rate by CQ type per turn:
  CQ1: {'ALEATORIC': '31.2%', 'EPISTEMIC': '75.0%'}
  CQ2: {'ALEATORIC': '73.3%', 'EPISTEMIC': '89.4%'}
  CQ3: {'ALEATORIC': '69.2%', 'EPISTEMIC': '90.8%'}

Conf delta (turn) when simulator gives INFORMATIVE vs HEDGE:
  CQ1: {'HEDGED_WITH_INFO': 5.28, 'INFORMATIVE': 7.69, 'PURE_HEDGE': 2.97}
  CQ2: {'HEDGED_WITH_INFO': 0.47, 'INFORMATIVE': 6.38, 'PURE_HEDGE': 0.8}
  CQ3: {'HEDGED_WITH_INFO': 7.32, 'INFORMATIVE': 6.83, 'PURE_HEDGE': 6.96}


Saved: mt_fig_simulator.png


### 4.7 Solution Quality Arc (ROUGE-L)
Does solution quality (ROUGE-L vs accepted answer) improve across the 4 checkpoints? Does more CQ rounds yield measurably better solutions alongside the confidence gain?

In [19]:
rouge_means_mt = [mt[col].mean() for _, col in rouge_ckpts]
rouge_labels_mt = [lbl for lbl, _ in rouge_ckpts]

print("=== Multi-Turn Solution Quality (ROUGE-L) ===")
for lbl, col in rouge_ckpts:
    print(f"  {lbl:<15}: {mt[col].mean():.4f}")
print(f"  Total Δ     : {mt['rouge_delta'].mean():+.4f}")
print(f"  Improved    : {(mt['rouge_delta'] > 0).sum()} / {len(mt)}")

# CQ type × ROUGE-L delta per turn
print()
print("ROUGE-L delta by CQ type per turn:")
rouge_turn_bounds = {
    1: ("rouge_prelim", "rouge_s1"),
    2: ("rouge_s1",     "rouge_s2"),
    3: ("rouge_s2",     "rouge_final"),
}
for t, (c_before, c_after) in rouge_turn_bounds.items():
    mt_t = mt[["id", c_before, c_after]].copy()
    mt_t["rouge_turn_delta"] = mt_t[c_after] - mt_t[c_before]
    sub_t = typed_turn[typed_turn["turn"] == t].merge(
        mt_t[["id", "rouge_turn_delta"]], on="id", how="left"
    )
    ep_r = sub_t[sub_t["cq_type"] == "EPISTEMIC"]["rouge_turn_delta"]
    al_r = sub_t[sub_t["cq_type"] == "ALEATORIC"]["rouge_turn_delta"]
    print(f"  Turn {t}: EPISTEMIC mean={ep_r.mean():+.4f} (n={len(ep_r)}) "
          f"| ALEATORIC mean={al_r.mean():+.4f} (n={len(al_r)})")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Panel 1: ROUGE-L arc across checkpoints
axes[0].plot(range(4), rouge_means_mt, "o-", lw=2, ms=8, color="darkorange")
for i, v in enumerate(rouge_means_mt):
    axes[0].text(i, v + 0.003, f"{v:.3f}", ha="center", fontsize=9)
axes[0].set_xticks(range(4))
axes[0].set_xticklabels(rouge_labels_mt, rotation=10)
axes[0].set_ylabel("Mean ROUGE-L F1")
axes[0].set_ylim(0, max(rouge_means_mt) * 1.35)
axes[0].set_title("ROUGE-L Progression (4 Checkpoints)")

# Panel 2: Dual-axis — confidence + ROUGE-L together
conf_means_mt = [mt[col].mean() for _, col in mt_ckpts]
ax_c = axes[1]
ax_r = ax_c.twinx()
l1, = ax_c.plot(range(4), conf_means_mt, "o-", lw=2, ms=7, color="steelblue", label="Confidence")
l2, = ax_r.plot(range(4), rouge_means_mt, "s--", lw=2, ms=7, color="darkorange", label="ROUGE-L")
ax_c.set_ylabel("Mean Confidence", color="steelblue")
ax_r.set_ylabel("Mean ROUGE-L F1", color="darkorange")
ax_c.set_xticks(range(4))
ax_c.set_xticklabels(rouge_labels_mt, rotation=10)
ax_c.set_title("Confidence vs ROUGE-L Arc")
ax_c.legend([l1, l2], ["Confidence", "ROUGE-L"], loc="lower right")

fig.suptitle(f"Multi-Turn Solution Quality — {MODEL_ID}", fontsize=13)
plt.tight_layout()
plt.savefig(OUTPUTS / "mt_fig_rouge.png", dpi=150)
plt.show()
print("Saved: mt_fig_rouge.png")

=== Multi-Turn Solution Quality (ROUGE-L) ===
  Preliminary    : 0.1344
  After CQ1      : 0.1349
  After CQ2      : 0.1301
  Final          : 0.1323
  Total Δ     : -0.0021
  Improved    : 54 / 100

ROUGE-L delta by CQ type per turn:
  Turn 1: EPISTEMIC mean=+0.0013 (n=84) | ALEATORIC mean=-0.0040 (n=16)
  Turn 2: EPISTEMIC mean=-0.0039 (n=85) | ALEATORIC mean=-0.0094 (n=15)
  Turn 3: EPISTEMIC mean=+0.0012 (n=87) | ALEATORIC mean=+0.0089 (n=13)


Saved: mt_fig_rouge.png


### 4.8 Answer Variation, Confidence & Accuracy Across Turns
How much does the model's *answer* change between consecutive turns? High variation means the additional CQ rounds substantially revise the solution.

We measure **answer variation** as 1 − ROUGE-L between consecutive solutions (0 = identical, 1 = completely different) and compare it with the **confidence change** and **accuracy change** (ROUGE-L vs accepted answer) at the same turn.

In [20]:
mt_av = mt.copy()

# Answer variation: 1 - ROUGE-L between consecutive solutions
solution_pairs = [
    ("preliminary_solution", "solution_1",    "var_turn1"),
    ("solution_1",           "solution_2",    "var_turn2"),
    ("solution_2",           "final_solution","var_turn3"),
]
for col_a, col_b, new_col in solution_pairs:
    mt_av[new_col] = mt_av.apply(
        lambda r, ca=col_a, cb=col_b: 1.0 - rouge_l(r[ca], r[cb]), axis=1
    )

var_cols        = ["var_turn1", "var_turn2", "var_turn3"]
var_labels      = ["Prelim->CQ1", "CQ1->CQ2", "CQ2->Final"]
conf_delta_cols = ["conf_delta_1", "conf_delta_2", "conf_delta_3"]
acc_delta_cols  = ["acc_delta_1",  "acc_delta_2",  "acc_delta_3"]

mt_av["conf_delta_1"] = mt_av["confidence_1"]    - mt_av["preliminary_confidence"]
mt_av["conf_delta_2"] = mt_av["confidence_2"]    - mt_av["confidence_1"]
mt_av["conf_delta_3"] = mt_av["final_confidence"] - mt_av["confidence_2"]
mt_av["acc_delta_1"]  = mt_av["rouge_s1"]    - mt_av["rouge_prelim"]
mt_av["acc_delta_2"]  = mt_av["rouge_s2"]    - mt_av["rouge_s1"]
mt_av["acc_delta_3"]  = mt_av["rouge_final"] - mt_av["rouge_s2"]

print("=== Multi-Turn Answer Variation, Confidence & Accuracy Change ===\n")
print(f"{'Turn':>15} {'Ans Variation':>15} {'Conf Delta':>12} {'Acc Delta (ROUGE-L)':>20}")
print("-" * 65)
var_means  = [mt_av[c].mean() for c in var_cols]
conf_means = [mt_av[c].mean() for c in conf_delta_cols]
acc_means  = [mt_av[c].mean() for c in acc_delta_cols]
for lbl, v, c, a in zip(var_labels, var_means, conf_means, acc_means):
    print(f"  {lbl:<13} {v:>15.4f} {c:>+12.2f} {a:>+20.4f}")

print("\nCorrelation between answer variation and conf/accuracy change:")
for t_idx, (lbl, vcol, ccol, acol) in enumerate(zip(var_labels, var_cols, conf_delta_cols, acc_delta_cols), 1):
    clean_v = mt_av[[vcol, ccol, acol]].dropna()
    r_cv, _ = pearsonr(clean_v[vcol], clean_v[ccol])
    r_av, _ = pearsonr(clean_v[vcol], clean_v[acol])
    print(f"  Turn {t_idx}:  variation <-> conf_delta r={r_cv:+.3f}  |  variation <-> acc_delta r={r_av:+.3f}")

# Scatter grid: variation vs conf/accuracy delta per turn
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for col_idx, (lbl, vcol, ccol, acol) in enumerate(zip(var_labels, var_cols, conf_delta_cols, acc_delta_cols)):
    clean_v = mt_av[[vcol, ccol, acol]].dropna()
    r_cv, _ = pearsonr(clean_v[vcol], clean_v[ccol])
    r_av, _ = pearsonr(clean_v[vcol], clean_v[acol])

    ax0 = axes[0][col_idx]
    ax0.scatter(clean_v[vcol], clean_v[ccol], alpha=0.55, s=35, color="steelblue", edgecolors="none")
    ax0.axhline(0, color="gray", lw=0.8, linestyle="--")
    ax0.set_title(f"{lbl}\nVar <-> Conf Delta  r={r_cv:+.3f}")
    ax0.set_xlabel("Answer Variation (1-ROUGE-L)")
    if col_idx == 0:
        ax0.set_ylabel("Confidence delta (pp)")

    ax1 = axes[1][col_idx]
    ax1.scatter(clean_v[vcol], clean_v[acol], alpha=0.55, s=35, color="darkorange", edgecolors="none")
    ax1.axhline(0, color="gray", lw=0.8, linestyle="--")
    ax1.set_title(f"{lbl}\nVar <-> Acc Delta  r={r_av:+.3f}")
    ax1.set_xlabel("Answer Variation (1-ROUGE-L)")
    if col_idx == 0:
        ax1.set_ylabel("ROUGE-L delta (accuracy)")

fig.suptitle(f"Answer Variation x Confidence & Accuracy Change - Multi-Turn ({MODEL_ID})", fontsize=13)
plt.tight_layout()
plt.savefig(OUTPUTS / "mt_fig_answer_variation.png", dpi=150)
plt.show()
print("Saved: mt_fig_answer_variation.png")

# Summary triple-axis overview
fig2, ax3 = plt.subplots(figsize=(8, 4))
x = [1, 2, 3]
ax3.plot(x, var_means,  "o-",  lw=2, ms=7, color="gray",       label="Answer variation (1-ROUGE-L)")
ax_r = ax3.twinx()
ax_r.plot(x, conf_means, "s--", lw=2, ms=7, color="steelblue",  label="Conf delta (pp)")
ax_r.plot(x, acc_means,  "^-.", lw=2, ms=7, color="darkorange", label="Acc delta (ROUGE-L)")
ax3.set_xticks(x)
ax3.set_xticklabels(var_labels)
ax3.set_ylabel("Answer Variation", color="gray")
ax_r.set_ylabel("Delta values")
ax3.set_title(f"Answer Variation, Conf Delta, Acc Delta per Turn - {MODEL_ID}")
lines1, labels1 = ax3.get_legend_handles_labels()
lines2, labels2 = ax_r.get_legend_handles_labels()
ax3.legend(lines1+lines2, labels1+labels2, fontsize=9, loc="upper left")
plt.tight_layout()
plt.savefig(OUTPUTS / "mt_fig_variation_summary.png", dpi=150)
plt.show()
print("Saved: mt_fig_variation_summary.png")


=== Multi-Turn Answer Variation, Confidence & Accuracy Change ===

           Turn   Ans Variation   Conf Delta  Acc Delta (ROUGE-L)
-----------------------------------------------------------------
  Prelim->CQ1            0.6343        +5.31              +0.0005
  CQ1->CQ2               0.5924        +1.40              -0.0048
  CQ2->Final             0.5775        +7.07              +0.0022

Correlation between answer variation and conf/accuracy change:
  Turn 1:  variation <-> conf_delta r=+0.128  |  variation <-> acc_delta r=-0.036
  Turn 2:  variation <-> conf_delta r=+0.122  |  variation <-> acc_delta r=-0.044
  Turn 3:  variation <-> conf_delta r=+0.090  |  variation <-> acc_delta r=-0.032


Saved: mt_fig_answer_variation.png


Saved: mt_fig_variation_summary.png


---
## 5. Cross-Experiment Comparison
*Single-turn vs Multi-turn on the same 100 cases*

In [21]:
# Restore vc/vc_mt in case earlier cells overwrote them
vc    = st_valid['label'].value_counts()
vc_mt = mt_valid['label'].value_counts()

st_delta       = st["updated_confidence"] - st["preliminary_confidence"]
mt_final_delta = mt["final_confidence"]   - mt["preliminary_confidence"]

print("=== Holistic Summary: Confidence & Solution Quality ===")
print(f"{'':30} {'Single-turn':>14} {'Multi-turn':>12}")
print("-" * 58)
print(f"  {'Preliminary confidence':28} {st['preliminary_confidence'].mean():>14.1f} {mt['preliminary_confidence'].mean():>12.1f}")
print(f"  {'Final confidence':28} {st['updated_confidence'].mean():>14.1f} {mt['final_confidence'].mean():>12.1f}")
print(f"  {'Confidence Δ mean':28} {st_delta.mean():>+14.1f} {mt_final_delta.mean():>+12.1f}")
print()
print(f"  {'Preliminary ROUGE-L':28} {st['rouge_prelim'].mean():>14.4f} {mt['rouge_prelim'].mean():>12.4f}")
print(f"  {'Final ROUGE-L':28} {st['rouge_updated'].mean():>14.4f} {mt['rouge_final'].mean():>12.4f}")
print(f"  {'ROUGE-L Δ mean':28} {st['rouge_delta'].mean():>+14.4f} {mt['rouge_delta'].mean():>+12.4f}")
print()
print(f"  {'EPISTEMIC CQ %':28} {100*vc.get('EPISTEMIC',0)/len(st_valid):>14.1f}% {100*vc_mt.get('EPISTEMIC',0)/len(mt_valid):>12.1f}%")

# ── 2-panel side-by-side comparison ───────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Panel 1: Confidence progression
axes[0].plot([0, 1],
             [st["preliminary_confidence"].mean(), st["updated_confidence"].mean()],
             "o-", lw=2.5, ms=8, color="steelblue", label="Single-turn")
axes[0].plot([0, 1, 2, 3],
             [mt[col].mean() for _, col in mt_ckpts],
             "s-", lw=2.5, ms=8, color="seagreen", label="Multi-turn")
axes[0].set_xticks([0, 1, 2, 3])
axes[0].set_xticklabels(["Preliminary", "After CQ1", "After CQ2", "Final"])
axes[0].set_ylabel("Mean Confidence")
axes[0].set_ylim(60, 100)
axes[0].set_title("Confidence Progression")
axes[0].legend()
axes[0].tick_params(axis="x", rotation=10)

# Panel 2: ROUGE-L progression
axes[1].plot([0, 1],
             [st["rouge_prelim"].mean(), st["rouge_updated"].mean()],
             "o-", lw=2.5, ms=8, color="steelblue", label="Single-turn")
axes[1].plot([0, 1, 2, 3],
             [mt[col].mean() for _, col in rouge_ckpts],
             "s-", lw=2.5, ms=8, color="seagreen", label="Multi-turn")
axes[1].set_xticks([0, 1, 2, 3])
axes[1].set_xticklabels(["Preliminary", "After CQ1", "After CQ2", "Final"])
axes[1].set_ylabel("Mean ROUGE-L F1")
axes[1].set_ylim(0, 0.35)
axes[1].set_title("Solution Quality (ROUGE-L) Progression")
axes[1].legend()
axes[1].tick_params(axis="x", rotation=10)

fig.suptitle(f"Holistic Comparison: Confidence + Quality — {MODEL_ID}", fontsize=13)
plt.tight_layout()
plt.savefig(OUTPUTS / "fig_comparison.png", dpi=150)
plt.show()
print("Saved: fig_comparison.png")

=== Holistic Summary: Confidence & Solution Quality ===
                                  Single-turn   Multi-turn
----------------------------------------------------------
  Preliminary confidence                 79.7         78.3
  Final confidence                       90.5         92.1
  Confidence Δ mean                     +10.9        +13.8

  Preliminary ROUGE-L                  0.1355       0.1344
  Final ROUGE-L                        0.1387       0.1323
  ROUGE-L Δ mean                      +0.0032      -0.0021

  EPISTEMIC CQ %                         82.0%         85.3%


Saved: fig_comparison.png


---
## 6. Export Summary CSV

In [22]:
# ── Single-turn summary ────────────────────────────────────────────────────────
# st_with_type already has rouge cols (from st via cell a0000012)
st_export = st_with_type[[
    "id", "category",
    "preliminary_confidence", "updated_confidence", "conf_delta",
    "clarifying_question", "cq_type",
    "rouge_prelim", "rouge_updated", "rouge_delta",
]].copy()

st_out = OUTPUTS / "st_analysis_summary.csv"
st_export.to_csv(st_out, index=False)
print(f"Single-turn summary: {st_out.name}  ({len(st_export)} rows)")

# ── Multi-turn summary ─────────────────────────────────────────────────────────
mt_export = mt[[
    "id", "category",
    "preliminary_confidence", "confidence_1", "confidence_2", "final_confidence",
    "cq_1", "cq_2", "cq_3",
    "rouge_prelim", "rouge_s1", "rouge_s2", "rouge_final", "rouge_delta",
]].copy()
mt_export["total_delta"] = mt_export["final_confidence"] - mt_export["preliminary_confidence"]

for t in [1, 2, 3]:
    turn_types = mt_valid[mt_valid["turn"] == t][["id", "label"]].rename(columns={"label": f"cq{t}_type"})
    mt_export = mt_export.merge(turn_types, on="id", how="left")

mt_out = OUTPUTS / "mt_analysis_summary.csv"
mt_export.to_csv(mt_out, index=False)
print(f"Multi-turn summary:  {mt_out.name}  ({len(mt_export)} rows)")
display(mt_export.head(3))

Single-turn summary: st_analysis_summary.csv  (100 rows)
Multi-turn summary:  mt_analysis_summary.csv  (100 rows)


,id,category,preliminary_confidence,confidence_1,confidence_2,final_confidence,cq_1,cq_2,cq_3,rouge_prelim,rouge_s1,rouge_s2,rouge_final,rouge_delta,total_delta,cq1_type,cq2_type,cq3_type
0,msd_001,Windows_8.1,90,90,95,98,Can you still find and launch Microsoft Solita...,Have you noticed any other icons missing from ...,"Could you please try searching for ""Microsoft ...",0.222222,0.198758,0.206061,0.226415,0.004193,8,EPISTEMIC,EPISTEMIC,EPISTEMIC
1,msd_002,Windows_8.1,90,95,98,99,Have you tried searching for 'calc.exe' or 'Ca...,Are you encountering any issues when trying to...,Are you familiar with how to right-click on an...,0.135922,0.143791,0.148867,0.129032,-0.006890,9,EPISTEMIC,EPISTEMIC,EPISTEMIC
2,msd_003,Bing_Apps,90,95,95,98,Does the Bing News app crash immediately upon ...,Have you recently installed any Windows update...,When you mentioned other users having the same...,0.134146,0.142077,0.133333,0.134831,0.000685,8,ALEATORIC,EPISTEMIC,EPISTEMIC
